In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install fastnode2vec

# Lib


In [ ]:
import os
import gc
import time
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from fastnode2vec import Graph, Node2Vec

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, roc_curve, confusion_matrix, ndcg_score, classification_report
from sklearn.calibration import CalibratedClassifierCV

import warnings
warnings.filterwarnings('ignore')

# Config

In [ ]:
config = {
    # Feature extraction
    'use_node_features': True,
    'use_edge_heuristics': True,
    'use_skill_similarity': True,

    # Node Embedding
    'use_node2vec': True,

    # Column names
    'source_col': 'source',
    'target_col': 'target',
    'label_col': 'label',

    # Path
    "train_path": "/content/drive/MyDrive/colab/Social/train.csv",
    "test_path": "/content/drive/MyDrive/colab/Social/test.csv",
    "feature_json_path": "/content/drive/MyDrive/colab/Social/musae_git_features.json"
}

# Func

## Graph

In [ ]:
def build_graph(train_df, test_df, source_col='source', target_col='target', label_col='label'):
    G = nx.Graph()
    train_nodes = set(train_df[source_col].unique()) | set(train_df[target_col].unique())
    test_nodes = set(test_df[source_col].unique()) | set(test_df[target_col].unique())
    all_nodes = train_nodes | test_nodes
    G.add_nodes_from(all_nodes)
    positive_edges = train_df[train_df[label_col] == 1][[source_col, target_col]].values
    G.add_edges_from(positive_edges)
    print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G

## Preprocessing

In [ ]:
def preprocessing(X_train, X_test):
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

## Feature extraction

In [ ]:
def extract_node_level_features(G, df, source_col='source', target_col='target', use=True):
    if not use:
        return pd.DataFrame(index=df.index)
    deg_cent = nx.degree_centrality(G)
    try:
        eig_cent = nx.eigenvector_centrality(G, max_iter=500, tol=1e-5)
    except nx.PowerIterationFailedConvergence:
        eig_cent = nx.eigenvector_centrality_numpy(G)
    pagerank = nx.pagerank(G, alpha=0.85, tol=1e-5)
    features = pd.DataFrame(index=df.index)
    centralities = [('deg', deg_cent), ('eig', eig_cent), ('pr', pagerank)]
    for name, cent_dict in centralities:
        features[f'{name}_source'] = df[source_col].map(cent_dict)
        features[f'{name}_target'] = df[target_col].map(cent_dict)
    print("[Extract] Completed extract node-level features")
    return features.fillna(0)

In [ ]:
def extract_edge_level_features(G, df, source_col='source', target_col='target', use=True):
    if not use:
        return pd.DataFrame(index=df.index)
    features = pd.DataFrame(index=df.index)
    ebunch = list(zip(df[source_col], df[target_col]))
    features['common_neighbors'] = [len(list(nx.common_neighbors(G, u, v))) for u, v in ebunch]
    jac_gen = nx.jaccard_coefficient(G, ebunch)
    features['jaccard'] = [val for _, _, val in jac_gen]
    aa_gen = nx.adamic_adar_index(G, ebunch)
    features['adamic_adar'] = [val for _, _, val in aa_gen]
    def sp_dist(u, v):
        try:
            return nx.shortest_path_length(G, u, v)
        except nx.NetworkXNoPath:
            return len(G.nodes()) + 1
    features['shortest_path'] = [sp_dist(u, v) for u, v in ebunch]
    print("[Extract] Completed extract edge-level features")
    return features

In [ ]:
def extract_skill_similarity(df, json_path, source_col='source', target_col='target', use=True, feature_prefix='skill'):
    if not use:
        return pd.DataFrame(index=df.index)
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_dict = json.load(f)
    str_entity_dict = {str(k): set(v) for k, v in raw_dict.items()}
    su_list = [str_entity_dict.get(str(x), set()) for x in df[source_col]]
    sv_list = [str_entity_dict.get(str(x), set()) for x in df[target_col]]
    inter_len = np.array([len(u & v) for u, v in zip(su_list, sv_list)])
    len_u = np.array([len(u) for u in su_list])
    len_v = np.array([len(v) for v in sv_list])
    union_len = len_u + len_v - inter_len
    features = pd.DataFrame(index=df.index)
    col_common = f'{feature_prefix}_common'
    col_jaccard = f'{feature_prefix}_jaccard'
    col_dice = f'{feature_prefix}_dice'
    col_cosine = f'{feature_prefix}_cosine'
    col_overlap = f'{feature_prefix}_overlap_ratio'
    features[col_common] = inter_len
    with np.errstate(divide='ignore', invalid='ignore'):
        features[col_jaccard] = np.where(union_len > 0, inter_len / union_len, 0.0)
        features[col_dice] = np.where((len_u + len_v) > 0, (2.0 * inter_len) / (len_u + len_v), 0.0)
        features[col_cosine] = np.where((len_u * len_v) > 0, inter_len / np.sqrt(len_u * len_v), 0.0)
        min_len = np.minimum(len_u, len_v)
        features[col_overlap] = np.where(min_len > 0, inter_len / min_len, 0.0)
    print(f"[Extract] Completed extract skill similarity")
    return features

## Graph embedding

### Node

In [ ]:
def get_node2vec_embeddings(G):
    print("[Embedding] Starting node embedding")
    is_weighted = any("weight" in edge_data for _, _, edge_data in G.edges(data=True))
    if is_weighted:
        edges = [(u, v, data.get("weight", 1.0)) for u, v, data in G.edges(data=True)]
    else:
        edges = list(G.edges())
    n2v_graph = Graph(
        edges,
        directed=G.is_directed(),
        weighted=is_weighted,
        number_of_edges=len(edges),
    )
    DIMENSIONS = 64
    model = Node2Vec(
        n2v_graph,
        dim=DIMENSIONS,
        walk_length=30,
        window=10,
        p=0.5,
        q=2.0,
        workers=4,
        batch_walks=10000,
        seed=42
    )
    model.train(epochs=10)
    actual_dim = model.wv.vector_size if hasattr(model, "wv") else DIMENSIONS
    embeddings = {}
    matched = 0
    missing = 0
    for node in G.nodes():
        if node in model.wv:
            embeddings[node] = np.asarray(model.wv[node], dtype=np.float32)
            matched += 1
        else:
            embeddings[node] = np.zeros(actual_dim, dtype=np.float32)
            missing += 1
    print("[Embedding] Node type:", type(next(iter(G.nodes()))))
    print("[Embedding] First graph nodes:", list(G.nodes())[:10])
    print("[Embedding] First vocab keys:", model.wv.index_to_key[:10])
    print("[Embedding] Embedding dim:", actual_dim)
    print("[Embedding] Matched:", matched)
    print("[Embedding] Missing:", missing)
    if matched > 0:
        sample_node = next(iter(G.nodes()))
        if sample_node in model.wv:
            print("[Embedding] Sample embedding:", model.wv[sample_node][:5])
    print("[Embedding] Completed node embeddings")

    return embeddings

### Edge

In [ ]:
def get_edge_embeddings(df, embeddings, operator='hadamard', source_col='source', target_col='target'):
    if embeddings is None:
        return pd.DataFrame(index=df.index)
    dim = len(next(iter(embeddings.values())))
    edge_emb_list = []
    for u, v in zip(df[source_col], df[target_col]):
        emb_u = np.array(embeddings.get(u, np.zeros(dim)))
        emb_v = np.array(embeddings.get(v, np.zeros(dim)))
        if operator == 'hadamard':
            edge_emb = emb_u * emb_v
        elif operator == 'l1':
            edge_emb = np.abs(emb_u - emb_v)
        elif operator == 'l2':
            edge_emb = np.square(emb_u - emb_v)
        elif operator == 'average':
            edge_emb = (emb_u + emb_v) / 2.0
        else:
            raise ValueError("Operator is not allowed")
        edge_emb_list.append(edge_emb)
    columns = [f'n2v_{operator}_{i}' for i in range(dim)]
    print(f"[Embedding] Completed edge embeddings by {operator}")
    return pd.DataFrame(edge_emb_list, columns=columns, index=df.index)

## Train

In [ ]:
def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    print(f"[Train] Completed")
    return model

## Load

In [ ]:
def load_model(model_save_path):
    loaded_model = joblib.load(model_save_path)
    print(f"[Load] Model loaded successfully from {model_save_path}")
    return loaded_model

# Main

## Load data

In [ ]:
train_df = pd.read_csv(config['train_path'])
test_df = pd.read_csv(config['test_path'])

## Build graph

In [ ]:
G = build_graph(train_df, test_df, source_col=config['source_col'], target_col=config['target_col'], label_col=config['label_col'])

## Feature extraction

In [ ]:
node_train = extract_node_level_features(G, train_df, config['source_col'], config['target_col'], config['use_node_features'])
edge_train = extract_edge_level_features(G, train_df, config['source_col'], config['target_col'], config['use_edge_heuristics'])
skill_train = extract_skill_similarity(train_df, config['feature_json_path'], config['source_col'], config['target_col'], config['use_skill_similarity'])

In [ ]:
node_test  = extract_node_level_features(G, test_df,  config['source_col'], config['target_col'], config['use_node_features'])
edge_test  = extract_edge_level_features(G, test_df,  config['source_col'], config['target_col'], config['use_edge_heuristics'])
skill_test  = extract_skill_similarity(test_df,  config['feature_json_path'], config['source_col'], config['target_col'], config['use_skill_similarity'])

In [ ]:
if config['use_node_features']:
  node_train, node_test, _ = preprocessing(node_train, node_test)
  node_train = pd.DataFrame(node_train, index=train_df.index)
  node_test = pd.DataFrame(node_test, index=test_df.index)

if config['use_edge_heuristics']:
  edge_train, edge_test, _ = preprocessing(edge_train, edge_test)
  edge_train = pd.DataFrame(edge_train, index=train_df.index)
  edge_test  = pd.DataFrame(edge_test,  index=test_df.index)

if config['use_skill_similarity']:
  skill_train, skill_test, _ = preprocessing(skill_train, skill_test)
  skill_train = pd.DataFrame(skill_train, index=train_df.index)
  skill_test  = pd.DataFrame(skill_test,  index=test_df.index)

## Node embeddings

In [ ]:
embeddings = None
if config['use_node2vec']:
    embeddings = get_node2vec_embeddings(G)

## Edge embeddings

In [ ]:
operator = 'hadamard'
n2v_train = get_edge_embeddings(df=train_df, embeddings=embeddings, operator=operator, source_col=config['source_col'], target_col=config['target_col'])
n2v_test  = get_edge_embeddings(df=test_df, embeddings=embeddings, operator=operator, source_col=config['source_col'], target_col=config['target_col'])

## Matrix features

In [ ]:
X_train = None
X_test = None
X_train = pd.concat([node_train, edge_train, skill_train, n2v_train], axis=1)
X_test  = pd.concat([node_test,  edge_test,  skill_test,  n2v_test],  axis=1)

In [ ]:
X_train = X_train.values
X_test = X_test.values
y_train = train_df[config['label_col']]
y_test  = test_df[config['label_col']]

# Model

## Train

In [ ]:
model_name = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

In [ ]:
model = train_model(model_name, X_train, y_train)

## Save

In [ ]:
model_save_path = '/content/drive/MyDrive/colab/Social/model/random_forest.joblib'
joblib.dump(model, model_save_path)
print(f"[Export] Model exported successfully to {model_save_path}")